# Notebook 08 - Fast Ensemble Kaggle (Phase 2)

Goal: generate several **fast** ensemble submissions by fusing existing ranked lists already produced in `../outputs`.

This avoids expensive retraining/re-embedding and lets you test multiple variants on Kaggle quickly.

In [1]:
import json
import os
from collections import defaultdict

import pandas as pd

OUTPUTS_DIR = '../outputs'
K_SUBMIT = 100

# Candidate runs already present in outputs/
RUN_FILES = {
    'v8_hybrid_cv': '92iemj_v8_hybrid_cv.csv',
    'phase2_final': 'submission_phase2_final.csv',
    'v7_tuned': '92iemj_v7_tuned.csv',
    'baseline_92': '92iemj.csv',
}

# Use categories from this file (already predicted, no '?')
CATEGORY_SOURCE = 'submission_phase2_final.csv'

print('OUTPUTS_DIR =', OUTPUTS_DIR)
print('Runs =', RUN_FILES)

OUTPUTS_DIR = ../outputs
Runs = {'v8_hybrid_cv': '92iemj_v8_hybrid_cv.csv', 'phase2_final': 'submission_phase2_final.csv', 'v7_tuned': '92iemj_v7_tuned.csv', 'baseline_92': '92iemj.csv'}


In [2]:
def load_submission(path):
    df = pd.read_csv(path)
    expected = {'query_id', 'relevant_doc_ids', 'category'}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f'{path} missing columns: {missing}')
    df = df.copy()
    df['relevant_doc_ids'] = df['relevant_doc_ids'].apply(json.loads)
    return df

loaded = {}
for run_name, file_name in RUN_FILES.items():
    p = os.path.join(OUTPUTS_DIR, file_name)
    if not os.path.exists(p):
        print(f'[WARN] missing: {p}')
        continue
    loaded[run_name] = load_submission(p)
    print(f'[OK] {run_name:14s} -> {file_name} ({len(loaded[run_name])} queries)')

if not loaded:
    raise RuntimeError('No candidate submissions found in outputs/.')

# Integrity: all runs must have same query_id order
base_name = next(iter(loaded))
base_ids = loaded[base_name]['query_id'].tolist()
for run_name, df_run in loaded.items():
    same = (df_run['query_id'].tolist() == base_ids)
    if not same:
        raise ValueError(f'Query order mismatch for run: {run_name}')

cat_path = os.path.join(OUTPUTS_DIR, CATEGORY_SOURCE)
cat_df = load_submission(cat_path)
cat_map = dict(zip(cat_df['query_id'], cat_df['category']))
print('Category source =', CATEGORY_SOURCE)

[OK] v8_hybrid_cv   -> 92iemj_v8_hybrid_cv.csv (141 queries)
[OK] phase2_final   -> submission_phase2_final.csv (141 queries)
[OK] v7_tuned       -> 92iemj_v7_tuned.csv (141 queries)
[OK] baseline_92    -> 92iemj.csv (141 queries)
Category source = submission_phase2_final.csv


In [3]:
def weighted_rrf(rankings, weights, k_rrf=60, topk=100):
    scores = defaultdict(float)
    for docs, w in zip(rankings, weights):
        for rank, doc_id in enumerate(docs, start=1):
            scores[doc_id] += float(w) * (1.0 / (k_rrf + rank))
    fused = sorted(scores.keys(), key=lambda d: scores[d], reverse=True)
    return fused[:topk]

# Several fast variants to test on Kaggle
CONFIGS = [
    {
        'name': 'submission_fast_ens_v8_p2f_equal.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final'],
        'weights': [1.0, 1.0],
        'k_rrf': 60,
    },
    {
        'name': 'submission_fast_ens_v8_p2f_2to1.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final'],
        'weights': [2.0, 1.0],
        'k_rrf': 20,
    },
    {
        'name': 'submission_fast_ens_v8_p2f_v7.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final', 'v7_tuned'],
        'weights': [2.0, 1.0, 0.5],
        'k_rrf': 20,
    },
    {
        'name': 'submission_fast_ens_all4.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final', 'v7_tuned', 'baseline_92'],
        'weights': [2.0, 1.0, 0.5, 0.5],
        'k_rrf': 20,
    },
]

for cfg in CONFIGS:
    missing_runs = [r for r in cfg['runs'] if r not in loaded]
    if missing_runs:
        print(f"[SKIP] {cfg['name']} missing runs: {missing_runs}")
        continue

    rows = []
    qids = loaded[cfg['runs'][0]]['query_id'].tolist()

    for i, qid in enumerate(qids):
        rankings = [loaded[r].iloc[i]['relevant_doc_ids'] for r in cfg['runs']]
        fused = weighted_rrf(rankings, cfg['weights'], k_rrf=cfg['k_rrf'], topk=K_SUBMIT)

        rows.append({
            'query_id': qid,
            'relevant_doc_ids': json.dumps(fused),
            'category': cat_map.get(qid, 'tex'),
        })

    out_path = os.path.join(OUTPUTS_DIR, cfg['name'])
    df_out = pd.DataFrame(rows)

    # format checks
    n_docs = df_out['relevant_doc_ids'].apply(lambda x: len(json.loads(x)))
    assert (n_docs == K_SUBMIT).all(), f'Invalid top-k in {cfg["name"]}'

    df_out.to_csv(out_path, index=False)
    print(f"[OK] {cfg['name']}  |  rows={len(df_out)}")

[OK] submission_fast_ens_v8_p2f_equal.csv  |  rows=141
[OK] submission_fast_ens_v8_p2f_2to1.csv  |  rows=141
[OK] submission_fast_ens_v8_p2f_v7.csv  |  rows=141
[OK] submission_fast_ens_all4.csv  |  rows=141


## Next step
Submit the generated files to Kaggle and keep the best score.

Recommended order (fastest check first):
1. `submission_fast_ens_v8_p2f_2to1.csv`
2. `submission_fast_ens_v8_p2f_equal.csv`
3. `submission_fast_ens_v8_p2f_v7.csv`
4. `submission_fast_ens_all4.csv`